# 3. Revisar predicciones del modelo seleccionado
Se muestran registros reservados del mismo dataset limpio de V0.1. No hace
falta crear un CSV de propuesta ni reconstruir datos desde archivos mensuales.
Esta es una prueba con resultados históricos conocidos, no una oferta nueva.

El modelo ahora prioriza precisión: marca menos registros como seleccionables
para reducir falsos positivos. Una predicción 0 no significa que sea imposible
ganar. El mínimo de detección se eligió en validación y no es una garantía futura.


Se usa la politica intermedia del notebook 02 (30% de deteccion minima en validacion por defecto), cuando esta guardada.

In [1]:
from pathlib import Path
import os
BASE = Path.cwd().resolve()
if not (BASE / "02_Comparacion_de_Modelos.ipynb").exists() and (BASE / "V0.2" / "02_Comparacion_de_Modelos.ipynb").exists():
    BASE = BASE / "V0.2"
if not (BASE / "02_Comparacion_de_Modelos.ipynb").exists():
    raise FileNotFoundError("Abre el proyecto desde la raíz o desde V0.2.")
os.chdir(BASE)
import pandas as pd
import numpy as np
import json
import joblib
import matplotlib.pyplot as plt
from IPython.display import display
FUENTE = BASE.parent / "V0.1" / "LICIT_depurando_sinNulos.feather"
SALIDA = BASE / "resultados_dataset_limpio"
SALIDA.mkdir(exist_ok=True)
print("Dataset de entrada:", FUENTE)

Dataset de entrada: D:\Licit\1\ML-SmartBids\V0.1\LICIT_depurando_sinNulos.feather


In [2]:
ruta_equilibrada = SALIDA / "modelo_cobertura_equilibrada.joblib"
ruta_elegida = ruta_equilibrada if ruta_equilibrada.exists() else SALIDA / "modelo_dataset_limpio.joblib"
artefacto = joblib.load(ruta_elegida)
print("Politica cargada:", ruta_elegida.name)
print("Modelo:", artefacto["informe"]["modelo"])
print("Umbral:", artefacto["umbral"])
ejemplos = pd.read_csv(SALIDA / "prueba_registros.csv")
ejemplos["prediccion"] = (ejemplos["probabilidad_seleccion"] >= artefacto["umbral"]).astype(int)
display(ejemplos)
print("Objetivo:", artefacto["informe"].get("objetivo", "anterior"))
display(pd.Series(artefacto["informe"]["test"]))

print("Minimo de deteccion en validacion:", artefacto["informe"].get("recall_minimo_validacion"))

Politica cargada: modelo_cobertura_equilibrada.joblib
Modelo: Red_Neuronal_MLP
Umbral: 0.21275579707016437


,CodigoUnidad,MontoEstimado,CodigoProductoONU,Cantidad,Cantidad Ofertada,Valor Total Ofertado,AnioPublicacion,MesPublicacion,DiaSemanaPublicacion,sector_FFAA,...,Region_Región de Magallanes y de la Antártica Chilena,Region_Región de Tarapacá,Region_Región de Valparaíso,Region_Región de Ñuble,Region_Región del Biobío,Region_Región del Libertador General Bernardo O'Higgins,Region_Región del Maule,Seleccionada_real,probabilidad_seleccion,prediccion
0,1604,20.285928,42221504,5.638355,5.638355,14.844130,2024,1,3,0,...,0,0,1,0,0,0,0,0,0.245061,1
1,1604,20.285928,42221504,4.795791,4.795791,15.384127,2024,1,3,0,...,0,0,1,0,0,0,0,0,0.239444,1
2,1604,20.285928,42221504,6.685861,6.685861,17.974394,2024,1,3,0,...,0,0,1,0,0,0,0,1,0.178340,0
3,1604,20.285928,42221504,1.791759,1.791759,12.100440,2024,1,3,0,...,0,0,1,0,0,0,0,0,0.151571,0
4,1604,20.285928,42221504,5.707110,5.707110,16.194779,2024,1,3,0,...,0,0,1,0,0,0,0,0,0.214907,1
5,1604,20.285928,42221504,5.707110,5.707110,16.194779,2024,1,3,0,...,0,0,1,0,0,0,0,0,0.214907,1
6,1604,20.285928,42221504,1.386294,1.386294,11.938200,2024,1,3,0,...,0,0,1,0,0,0,0,0,0.141501,0
7,1604,20.285928,42221504,2.564949,2.564949,12.975905,2024,1,3,0,...,0,0,1,0,0,0,0,0,0.184663,0
8,1604,20.285928,42221504,2.564949,2.564949,12.975905,2024,1,3,0,...,0,0,1,0,0,0,0,0,0.184663,0
9,1604,20.285928,42221504,2.564949,2.564949,12.975905,2024,1,3,0,...,0,0,1,0,0,0,0,0,0.184663,0


Objetivo: precision


F1                                                     0.296303
precision                                              0.292283
recall                                                 0.300435
ROC_AUC                                                0.646633
average_precision                                       0.27129
Brier                                                  0.134782
log_loss                                               0.435428
matriz_confusion_orden_0_1    [[223372, 38601], [37121, 15942]]
dtype: object

Minimo de deteccion en validacion: 0.3


## Recalcular predicciones
El archivo de ejemplos contiene las mismas transformaciones de entrenamiento:
montos/cantidades en log(1+x), códigos categóricos y columnas one-hot heredadas.
No vuelvas a aplicar logaritmos a estos ejemplos ni alimentes el pipeline con
un CSV sin preparar. Para evaluar datos nuevos necesitas reproducir la preparación
de V0.1 y la función preparar_limpio; este artefacto no incluye esa cadena anterior.

In [3]:
entradas = ejemplos[artefacto["columnas"]].copy()
# Restauramos códigos como categorías, aunque read_csv los detecte como enteros.
for c in ["CodigoUnidad", "CodigoProductoONU"]:
    if c in entradas:
        entradas[c] = entradas[c].astype(str)
probabilidades = artefacto["modelo"].predict_proba(entradas)[:, 1]
vista = pd.DataFrame({"Seleccionada_real": ejemplos.Seleccionada_real,
                      "probabilidad": probabilidades,
                      "prediccion": (probabilidades >= artefacto["umbral"]).astype(int)})
display(vista)
assert np.allclose(probabilidades, ejemplos.probabilidad_seleccion)
print("Las predicciones coinciden con las guardadas.")

,Seleccionada_real,probabilidad,prediccion
0,0,0.245061,1
1,0,0.239444,1
2,1,0.178340,0
3,0,0.151571,0
4,0,0.214907,1
5,0,0.214907,1
6,0,0.141501,0
7,0,0.184663,0
8,0,0.184663,0
9,0,0.184663,0


Las predicciones coinciden con las guardadas.


Las métricas y limitaciones se muestran en el notebook 02 y están incluidas en el modelo guardado.
Sin claves y fechas exactas no se puede asegurar independencia por licitación
ni rendimiento en licitaciones futuras. El objetivo sigue siendo Seleccionada;
no se evalúan automáticamente calidad técnica, cumplimiento o rentabilidad.